# Breast Cancer Diagnosis
## Import packages/modules

In [ ]:
from mistic import MixedKernel, combined_rank, cvSet, kernelWrapper, paramSet, perDiff, score_svc, svmSet

import matplotlib.pyplot as plt   
import numpy as np
import pandas as pd
import copy
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

## Import dataset

In [ ]:
# Import dataset
data_path = Path("wdbc.data")
if not data_path.exists():
    data_path = Path("mistic/examples/wdbc.data")
dataTable = pd.read_csv(data_path, sep = ",",header=None) 
properties = ["Radius","Texture","Perimeter","Area","Smoothness","Compactness","Concavity","Concave Points","Symmetry","Fractal Dimension"]
names = ["ID", "Diagnosis"]+[p+" mean" for p in properties]+[p+" SE" for p in properties]+[p+" worst" for p in properties]
dataTable.columns = pd.Index(names)
dataTable

## Scale data and setup test set

In [ ]:
X = dataTable.copy()
X.drop(columns=['ID','Diagnosis'],inplace=True)
y_all = dataTable.iloc[:,1]

X_train, X_test, y_train, y_test = train_test_split(X, y_all, 
                                                    stratify = y_all, 
                                                    random_state= 0, 
                                                    test_size= 1/5)

scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

## Define cross validation sets

In [ ]:
cv_set = cvSet(X = X_train, y = y_train.values)
cv_set.classification(num_sets = 5)

## Initialize SVC model and parameter grids

In [ ]:
svc = SVC(kernel = 'precomputed', class_weight="balanced", probability=False, tol = 1e-9)

C_range = [2**x for x in range(-2,5)]
gamma_range = [2**x for x in range(-9,1)]

parameter_grid = []
for cost in C_range:
    for gamma in gamma_range:
        parameter_grid.append(paramSet(model={"C": cost},kernel={"gamma": gamma}))

## Feature Selection

In [ ]:
kernel_type = "rbf"

ensemble = svmSet(svc, cv_set,score_method = score_svc().score,
                              kernel = MixedKernel.weighted_sum(
                                  [kernelWrapper(kernel_type, name=kernel_type)], weights=[1.0]),
                              kernel_feature_selection = "shared",
                              feature_set_policy = "per_model")

ensemble.greedy_forward_selection(parameter_grid = parameter_grid,
                                   addition_factor = 0.1,
                                   feature_ranker = combined_rank(weight=0.95).compute,
                                   set_for_rank = "sample",
                                   tune_models_each_step = False)

## Pairplots and scatter plot of top 3 features

In [ ]:
import plotly.express as px
from seaborn import pairplot, clustermap, load_dataset

top_features = X.columns[ensemble.unified_sorted_features.astype(int)[0:3]].values
fig = pairplot(dataTable[np.append(top_features, 'Diagnosis')],
         hue="Diagnosis", corner=True, palette={"M":"red","B":"blue"})


In [ ]:
fig = px.scatter_3d(dataTable, x=top_features[0], y=top_features[1], z=top_features[2],
          color='Diagnosis')
fig.update_traces(marker_size = 4)
fig.show()

## Integrated Gradients Atrributions

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, leaves_list, fcluster

linkMeth = "average"

ig_result = ensemble.explain_integrated_gradients(
    X_test,
    feature_names=X.columns,
    target=np.asarray(y_test),
    model_index=None,
)

IGtable = ig_result.to_frame()

distMatCol = perDiff(IGtable)
hcCol = linkage(distMatCol, method=linkMeth,optimal_ordering=True)

tr_IGtable = IGtable.transpose()
distMatRow = perDiff(tr_IGtable)
hcRow = linkage(distMatRow, method=linkMeth,optimal_ordering=True)

color_val = max(np.abs((np.mean(ig_result.values)-np.std(ig_result.values), np.mean(ig_result.values)+np.std(ig_result.values))))
clustermap(IGtable, 
           yticklabels=False,
           cmap="vlag", vmin=-color_val, vmax=color_val,
           row_linkage=hcRow, col_linkage=hcCol, 
           row_colors=[["red","blue"][int(l=="B")+0] for l in y_test])

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
import numpy as np

features = ensemble.unified_features
df_X_test = pd.DataFrame(X_test[:,features],columns = IGtable.columns)
N = len(IGtable)

fig = go.Figure()
fig.update_layout(
    plot_bgcolor='white',
    autosize=False,
    width=800,
    height=800,
    coloraxis = {'colorscale':'Bluered'},
    xaxis_title = "Integrated Gradient",
)

fig.update_xaxes(
    zeroline=True,
    zerolinecolor="black",
    )
    
fig.update_yaxes(
    zeroline=True,
    zerolinecolor="black",
    mirror=True,
    ticks='outside',
    showline=True,
    linecolor='black',
    gridcolor='lightgrey'
)

feature_order = np.argsort(np.sum(abs(ig_result.values),axis=0))

y_val = 1
for f in feature_order:
    fig.add_trace(go.Scatter(
        x = IGtable.iloc[:,f], 
        y = y_val + np.random.rand(N)*0.5 - 0.25,
        mode = 'markers',
        marker = dict(size=9,
                      color = df_X_test.iloc[:,f],
                      coloraxis = "coloraxis",
                     ),
        name=IGtable.columns[f],
    ))
    y_val += 1


fig.update_layout(showlegend=False, coloraxis_showscale=True)
fig.update_coloraxes(colorbar_showticklabels=False,
                    colorbar_title=dict(text="Feature Value",side = "right"),
                    cmin = -1, cmax = 2)
fig.update_yaxes(tickvals=[i for i in range(1,IGtable.shape[1]+1)], ticktext=IGtable.columns[feature_order])
fig.show() 

In [ ]:
feat_1 = "Concave Points mean"
feat_2 = "Texture worst"

fig = px.scatter(x = df_X_test[feat_1], y = IGtable[feat_1], color = df_X_test[feat_2])
fig.update_traces(marker=dict(size=12))

fig.update_layout(
    plot_bgcolor='white',
    autosize=False,
    width=800,
    height=800,
    coloraxis = {'colorscale':'Bluered'},
    xaxis_title = feat_1,
    yaxis_title = "Integrated Gradient for " + feat_1,
)

fig.update_xaxes(
    zeroline=True,
    zerolinecolor='lightgrey',
    mirror=True,
    ticks='outside',
    showline=True,
    linecolor='black',
    gridcolor='lightgrey'
    )
    
fig.update_yaxes(
    zeroline=True,
    zerolinecolor='lightgrey',
    mirror=True,
    ticks='outside',
    showline=True,
    linecolor='black',
    gridcolor='lightgrey'
)

fig.update_coloraxes(colorbar_showticklabels=False,
                    colorbar_title=dict(text=feat_2,side = "right"),
                    cmin = -1, cmax = 2)

fig.show()